# Jakarta Urban Network Analysis with OSMnx

Analysis of street networks and amenities for **Jakarta, Indonesia** using OSMnx.

**Reference:** Boeing, G. 2017. "OSMnx: New Methods for Acquiring, Constructing, Analyzing, and Visualizing Complex Street Networks." *Computers, Environment and Urban Systems* 65, 126-139.

**City Info:**
- Population: ~10.5 million (metro: ~34 million)
- Area: ~662 km²
- Traffic segments: 14,549

⚠️ **Note:** Jakarta is a large network. Some computations may take longer.

## 1. Setup

In [ ]:
# Import libraries
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

# Configure OSMnx
ox.settings.use_cache = True
ox.settings.log_console = False

# City configuration
CITY = 'Jakarta'
BBOX = (-6.0911, -6.4096, 107.11, 106.6036)  # (north, south, east, west)
CENTER = (-6.250, 106.857)
COLOR = '#e74c3c'

print(f"Analyzing: {CITY}")
print(f"Bounding box: {BBOX}")

## 2. Download Street Network

In [ ]:
# Download street network (may take a few minutes for Jakarta)
print(f"Downloading {CITY} street network (this may take a few minutes)...")
G = ox.graph_from_bbox(bbox=BBOX, network_type='drive', simplify=True)
print(f"Nodes: {len(G.nodes):,}")
print(f"Edges: {len(G.edges):,}")

In [ ]:
# Visualize network
fig, ax = ox.plot_graph(G, node_size=0, edge_linewidth=0.3, edge_color=COLOR,
                         bgcolor='white', figsize=(14, 14), show=False, close=False)
ax.set_title(f'{CITY} Street Network\n({len(G.nodes):,} nodes, {len(G.edges):,} edges)',
             fontsize=14, fontweight='bold')
plt.savefig(f'figures/osm_{CITY.lower()}_network.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 3. Basic Network Statistics

In [ ]:
# Calculate basic statistics
stats = ox.basic_stats(G)

# Calculate area
lat_diff = abs(BBOX[1] - BBOX[0])
lon_diff = abs(BBOX[3] - BBOX[2])
area_km2 = lat_diff * 111 * lon_diff * 111 * np.cos(np.radians(np.mean([BBOX[0], BBOX[1]])))

print(f"\n{CITY} Network Statistics")
print("=" * 40)
print(f"Nodes: {stats['n']:,}")
print(f"Edges: {stats['m']:,}")
print(f"Area: {area_km2:.1f} km²")
print(f"Street density: {stats['street_density_km']:.2f} km/km²")
print(f"Node density: {stats['node_density_km']:.1f} nodes/km²")
print(f"Edge density: {stats['edge_density_km']:.1f} edges/km²")
print(f"Avg street length: {stats['street_length_avg']:.1f} m")
print(f"Total street length: {stats['street_length_total']/1000:.1f} km")
print(f"Avg streets per node: {stats['streets_per_node_avg']:.2f}")

## 4. Node Degree Analysis (Intersection Types)

In [ ]:
# Analyze node degrees
degrees = dict(G.degree())
degree_counts = pd.Series(degrees).value_counts().sort_index()

print(f"\nNode Degree Distribution")
print("=" * 40)
print(f"Dead ends (degree 1): {sum(1 for d in degrees.values() if d == 1):,}")
print(f"2-way nodes: {sum(1 for d in degrees.values() if d == 2):,}")
print(f"3-way intersections: {sum(1 for d in degrees.values() if d == 3):,}")
print(f"4-way intersections: {sum(1 for d in degrees.values() if d == 4):,}")
print(f"5+ way intersections: {sum(1 for d in degrees.values() if d >= 5):,}")
print(f"Mean degree: {np.mean(list(degrees.values())):.2f}")

In [ ]:
# Plot degree distribution
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(degree_counts.index, degree_counts.values, color=COLOR, alpha=0.7, edgecolor='black')
ax.axvline(np.mean(list(degrees.values())), color='blue', linestyle='--', 
           label=f'Mean: {np.mean(list(degrees.values())):.2f}')
ax.set_xlabel('Node Degree (number of connections)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title(f'{CITY} - Node Degree Distribution', fontsize=14, fontweight='bold')
ax.legend()
ax.set_xlim(0, 10)
plt.savefig(f'figures/osm_{CITY.lower()}_degree_dist.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 5. Centrality Analysis

⚠️ **Note:** For large networks like Jakarta, betweenness centrality can take a long time. We'll use sampling or skip if too slow.

In [ ]:
# For large networks, use approximate betweenness with k samples
print("Calculating approximate betweenness centrality (using sampling)...")
print("This may take several minutes for Jakarta...")

# Sample k nodes for approximation (faster for large networks)
k = min(500, len(G.nodes))  # Use 500 samples or all nodes if smaller
edge_bc = nx.edge_betweenness_centrality(G, normalized=True, k=k)
nx.set_edge_attributes(G, edge_bc, 'betweenness')
print(f"Done! Max betweenness: {max(edge_bc.values()):.6f}")

In [ ]:
# Plot betweenness centrality
fig, ax = plt.subplots(figsize=(16, 14))

edge_colors = [edge_bc.get((u, v), edge_bc.get((v, u), 0)) for u, v, k in G.edges(keys=True)]

ox.plot_graph(G, ax=ax, node_size=0, edge_linewidth=1,
              edge_color=edge_colors, edge_cmap=plt.cm.hot_r,
              bgcolor='white', show=False, close=False)

ax.set_title(f'{CITY} - Edge Betweenness Centrality\n(Critical routes for network flow)',
             fontsize=14, fontweight='bold')

sm = plt.cm.ScalarMappable(cmap=plt.cm.hot_r, norm=plt.Normalize(vmin=min(edge_colors), vmax=max(edge_colors)))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.7)
cbar.set_label('Betweenness Centrality', fontsize=11)

plt.savefig(f'figures/osm_{CITY.lower()}_betweenness.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Calculate closeness centrality (also sampled for speed)
print("Calculating closeness centrality...")

# For large graphs, sample nodes
sample_nodes = list(G.nodes())[:min(5000, len(G.nodes()))]
G_sample = G.subgraph(sample_nodes).copy()

closeness = nx.closeness_centrality(G_sample)
print(f"Mean closeness (sampled): {np.mean(list(closeness.values())):.6f}")

In [ ]:
# Plot closeness centrality (sampled)
fig, ax = plt.subplots(figsize=(16, 14))

node_colors = [closeness.get(node, 0) for node in G_sample.nodes()]

ox.plot_graph(G_sample, ax=ax, node_size=5, node_color=node_colors,
              node_cmap=plt.cm.viridis, edge_linewidth=0.2, edge_color='gray',
              bgcolor='white', show=False, close=False)

ax.set_title(f'{CITY} - Closeness Centrality (Sampled)\n(Accessibility to all other nodes)',
             fontsize=14, fontweight='bold')

sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, norm=plt.Normalize(vmin=min(node_colors), vmax=max(node_colors)))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.7)
cbar.set_label('Closeness Centrality', fontsize=11)

plt.savefig(f'figures/osm_{CITY.lower()}_closeness.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 6. Street Orientation Analysis

In [ ]:
# Calculate and plot street orientations
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})

G_bearing = ox.bearing.add_edge_bearings(G)
bearings = pd.Series([d['bearing'] for u, v, k, d in G_bearing.edges(keys=True, data=True)])

n_bins = 36
bins = np.linspace(0, 360, n_bins + 1)
counts, _ = np.histogram(bearings, bins=bins)

angles = np.deg2rad(bins[:-1])
width = np.deg2rad(360 / n_bins)

ax.bar(angles, counts, width=width, color=COLOR, alpha=0.7, edgecolor='black', linewidth=0.5)
ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)
ax.set_title(f'{CITY} - Street Orientation', fontsize=14, fontweight='bold', pad=20)

plt.savefig(f'figures/osm_{CITY.lower()}_orientation.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 7. Amenity Analysis

In [ ]:
# Download amenities
print(f"Downloading amenities for {CITY}...")
amenities = ox.features_from_bbox(bbox=BBOX, tags={'amenity': True})
print(f"Found {len(amenities)} amenities")

In [ ]:
# Count amenity types
if 'amenity' in amenities.columns:
    amenity_counts = amenities['amenity'].value_counts()
    print(f"\nTop 15 Amenity Types in {CITY}:")
    print(amenity_counts.head(15))

In [ ]:
# Plot amenity distribution
fig, ax = plt.subplots(figsize=(10, 8))

amenity_counts.head(15).plot(kind='barh', ax=ax, color=COLOR, edgecolor='black')
ax.set_xlabel('Count', fontsize=12)
ax.set_ylabel('Amenity Type', fontsize=12)
ax.set_title(f'{CITY} - Top 15 Amenity Types', fontsize=14, fontweight='bold')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(f'figures/osm_{CITY.lower()}_amenities.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Map amenities on network
fig, ax = plt.subplots(figsize=(16, 14))

ox.plot_graph(G, ax=ax, node_size=0, edge_linewidth=0.2, edge_color='gray',
              bgcolor='white', show=False, close=False)

amenities_points = amenities[amenities.geometry.type == 'Point']
if len(amenities_points) > 0:
    amenities_points.plot(ax=ax, color='red', markersize=3, alpha=0.5, label='Amenities')

ax.set_title(f'{CITY} - Street Network with Amenities', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')

plt.savefig(f'figures/osm_{CITY.lower()}_network_amenities.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 8. Integration with Traffic Data

In [ ]:
# Load traffic data
traffic = gpd.read_file(f'traffic_jkt_output/evening_peak_jkt.gpkg')
print(f"Traffic segments: {len(traffic)}")
print(f"Mean jam factor: {traffic['jam_factor_mean'].mean():.3f}")

In [ ]:
# Compare OSM network vs Traffic data
fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# OSM Network
edges = ox.graph_to_gdfs(G, nodes=False)
edges.plot(ax=axes[0], color='gray', linewidth=0.3)
axes[0].set_title(f'{CITY} - OSM Network\n({len(edges):,} edges)', fontsize=12, fontweight='bold')
axes[0].set_axis_off()

# Traffic Data
traffic.plot(column='jam_factor_mean', cmap='RdYlGn_r', linewidth=0.5, ax=axes[1],
             legend=True, legend_kwds={'label': 'Jam Factor', 'shrink': 0.7})
axes[1].set_title(f'{CITY} - Traffic Data\n({len(traffic):,} segments)', fontsize=12, fontweight='bold')
axes[1].set_axis_off()

plt.suptitle(f'{CITY} - OSM Network vs HERE Traffic Data', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'figures/osm_{CITY.lower()}_vs_traffic.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 9. Traffic Hotspots vs Network Centrality

In [ ]:
# Identify high-centrality edges and compare with traffic congestion
# Get top 10% highest betweenness edges
bc_threshold = np.percentile(list(edge_bc.values()), 90)
high_bc_edges = [(u, v) for (u, v), bc in edge_bc.items() if bc >= bc_threshold]

print(f"High centrality edges (top 10%): {len(high_bc_edges)}")
print(f"Betweenness threshold: {bc_threshold:.6f}")

In [ ]:
# Plot high centrality routes
fig, ax = plt.subplots(figsize=(16, 14))

# Plot all edges in gray
edges_gdf = ox.graph_to_gdfs(G, nodes=False)
edges_gdf.plot(ax=ax, color='lightgray', linewidth=0.3)

# Highlight high centrality edges
high_bc_gdf = edges_gdf.loc[edges_gdf.index.isin(high_bc_edges)]
if len(high_bc_gdf) > 0:
    high_bc_gdf.plot(ax=ax, color='red', linewidth=1.5, label='High Centrality Routes')

ax.set_title(f'{CITY} - Critical Routes (Top 10% Betweenness Centrality)',
             fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.set_axis_off()

plt.savefig(f'figures/osm_{CITY.lower()}_critical_routes.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 10. Export Results

In [ ]:
# Export network as GeoPackage
print("Exporting network (this may take a moment for Jakarta)...")
nodes, edges = ox.graph_to_gdfs(G)
filename = f'osm_network_{CITY.lower()}.gpkg'
edges.to_file(filename, driver='GPKG', layer='edges')
nodes.to_file(filename, driver='GPKG', layer='nodes')
print(f"Saved: {filename}")

In [ ]:
# Export statistics
stats_export = {
    'City': CITY,
    'Nodes': stats['n'],
    'Edges': stats['m'],
    'Area_km2': round(area_km2, 2),
    'Street_Density_km_per_km2': round(stats['street_density_km'], 2),
    'Node_Density_per_km2': round(stats['node_density_km'], 2),
    'Avg_Street_Length_m': round(stats['street_length_avg'], 2),
    'Total_Street_Length_km': round(stats['street_length_total']/1000, 2),
    'Mean_Node_Degree': round(np.mean(list(degrees.values())), 2),
    'Dead_Ends': sum(1 for d in degrees.values() if d == 1),
    'Three_Way_Intersections': sum(1 for d in degrees.values() if d == 3),
    'Four_Way_Intersections': sum(1 for d in degrees.values() if d == 4)
}

pd.DataFrame([stats_export]).to_csv(f'osm_stats_{CITY.lower()}.csv', index=False)
print(f"Saved: osm_stats_{CITY.lower()}.csv")

## Summary

This notebook analyzed the **Jakarta** urban street network using OSMnx, including:

- Street network topology and statistics
- Node degree distribution (intersection types)
- Betweenness and closeness centrality (sampled for performance)
- Street orientation patterns
- Amenity distribution
- Comparison with HERE traffic data
- Critical route identification

**Key Observations:**
- Jakarta has the largest network among the three cities
- High-centrality routes may correlate with traffic congestion
- Dense amenity distribution in central areas

**Reference:** Boeing, G. 2017. "OSMnx: New Methods for Acquiring, Constructing, Analyzing, and Visualizing Complex Street Networks."